![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/header_workshop.png)

### Seu database (schema no Unity Catalog)

Cada participante usa um **database** próprio. No Unity Catalog, isso é o **nome do schema** dentro do catálogo `dbacademy`.

**Regra:** **primeira letra do seu nome** + **seu sobrenome**, em **minúsculas**, sem espaços.

| Exemplo | Database |
|---------|----------|
| Aluno: **Carlos Bettanim** | `cbettanim` |

**Anote o seu identificador** — você vai usá-lo em **todos** os notebooks e referências.

Nos códigos SQL/Python deste notebook, substitua **`<seu_database>`** pelo seu valor (ex.: `cbettanim`). O objeto fica no formato `dbacademy.<seu_database>` → `dbacademy.cbettanim`.

> **Dica:** no Databricks, use *Edit → Find and Replace* no notebook para trocar `<seu_database>` de uma vez.

**Importante:** enquanto `<seu_database>` não for substituído por um nome válido (ex.: `cbettanim`), as células SQL/Python **não** devem ser executadas — o catálogo `dbacademy.<seu_database>` não é um identificador válido no Unity Catalog.


   

 Item | Descrição |
 --- | --- |
 **Objetivo** | Criar diferentes Genie Spaces |
 **Databricks Run Time** | DBR 16.4 LTS |
 **Linguagem** | SQL |

![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/03_diagram.png)

   
## Criar Genie Spaces

   
![](https://raw.githubusercontent.com/Databricks-BR/workshop_agents/refs/heads/main/demo-main/img/img/03_genie.png)

**Genie** é uma ferramenta de análise de dados conversacional que permite aos usuários interagir intuitivamente com seus dados por meio de linguagem natural, possibilitando que eles efetivamente "conversem com seus dados". Ao traduzir perguntas do dia a dia em consultas complexas e retornar insights acionáveis em tempo real, o Genie remove barreiras técnicas tradicionais, simplificando o processo de análise para usuários de todos os níveis de habilidade.

Essa abordagem não apenas acelera a tomada de decisões orientada por dados, mas também democratiza o acesso à inteligência de negócios, permitindo que qualquer pessoa explore, analise e visualize dados rapidamente por meio de interações simples baseadas em diálogo.

In [0]:
%sql
    
USE CATALOG dbacademy;
USE SCHEMA <seu_database>;
CREATE OR REPLACE FUNCTION _genie_query(databricks_host STRING, 
                  databricks_token STRING,
                  space_id STRING,
                  question STRING,
                  contextual_history STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Este é um agente com o qual você pode conversar para obter respostas a perguntas. Tente fazer perguntas simples e forneça histórico se houve conversas anteriores.'
AS
$$
    import json
    import os
    import time
    from dataclasses import dataclass
    from datetime import datetime
    from typing import Optional
    
    import pandas as pd
    import requests
    
    
    @dataclass
    class GenieResult:
        space_id: str
        conversation_id: str
        question: str
        content: Optional[str]
        sql_query: Optional[str] = None
        sql_query_description: Optional[str] = None
        sql_query_result: Optional[pd.DataFrame] = None
        error: Optional[str] = None
    
        def to_json_results(self):
            result = {
                "space_id": self.space_id,
                "conversation_id": self.conversation_id,
                "question": self.question,
                "content": self.content,
                "sql_query": self.sql_query,
                "sql_query_description": self.sql_query_description,
                "sql_query_result": self.sql_query_result.to_dict(
                    orient="records") if self.sql_query_result is not None else None,
                "error": self.error,
            }
            jsonified_results = json.dumps(result)
            return f"Resultados do Genie: {jsonified_results}"
    
        def to_string_results(self):
            results_string = self.sql_query_result.to_dict(orient="records") if self.sql_query_result is not None else None
            return ("Resultados do Genie: \n"
                    f"ID do Espaço: {self.space_id}\n"
                    f"ID da Conversa: {self.conversation_id}\n"
                    f"Pergunta Realizada: {self.question}\n"
                    f"Conteúdo: {self.content}\n"
                    f"Consulta SQL: {self.sql_query}\n"
                    f"Descrição da Consulta SQL: {self.sql_query_description}\n"
                    f"Resultado da Consulta SQL: {results_string}\n"
                    f"Erro: {self.error}")
    
    class GenieClient:
    
        def __init__(self, *,
                     host: Optional[str] = None,
                     token: Optional[str] = None,
                     api_prefix: str = "/api/2.0/genie/spaces"):
            self.host = host or os.environ.get("DATABRICKS_HOST")
            self.token = token or os.environ.get("DATABRICKS_TOKEN")
            assert self.host is not None, "DATABRICKS_HOST não está definido"
            assert self.token is not None, "DATABRICKS_TOKEN não está definido"
            self._workspace_client = requests.Session()
            self._workspace_client.headers.update({"Authorization": f"Bearer {self.token}"})
            self._workspace_client.headers.update({"Content-Type": "application/json"})
            self.api_prefix = api_prefix
            self.max_retries = 300
            self.retry_delay = 1
            self.new_line = "\r\n"
    
        def _make_url(self, path):
            return f"{self.host.rstrip('/')}/{path.lstrip('/')}"
    
        def start(self, space_id: str, start_suffix: str = "") -> str:
            path = self._make_url(f"{self.api_prefix}/{space_id}/start-conversation")
            resp = self._workspace_client.post(
                url=path,
                headers={"Content-Type": "application/json"},
                json={"content": "iniciando conversa" if not start_suffix else f"iniciando conversa {start_suffix}"},
            )
            resp = resp.json()
            print(resp)
            try:
              return resp["conversation_id"]
            except Exception:
              return resp
    
        def ask(self, space_id: str, conversation_id: str, message: str) -> GenieResult:
            path = self._make_url(f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages")
            resp_raw = self._workspace_client.post(
                url=path,
                headers={"Content-Type": "application/json"},
                json={"content": message},
            )
            resp = resp_raw.json()
            message_id = resp.get("message_id", resp.get("id"))
            if message_id is None:
                print(resp, resp_raw.url, resp_raw.status_code, resp_raw.headers)
                return GenieResult(content=None, error="Falha ao obter message_id")
    
            attempt = 0
            query = None
            query_description = None
            content = None
    
            while attempt < self.max_retries:
                resp_raw = self._workspace_client.get(
                    self._make_url(f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}"),
                    headers={"Content-Type": "application/json"},
                )
                resp = resp_raw.json()
                status = resp["status"]
                if status == "COMPLETED":
                    try:
    
                        query = resp["attachments"][0]["query"]["query"]
                        query_description = resp["attachments"][0]["query"].get("description", None)
                        content = resp["attachments"][0].get("text", {}).get("content", None)
                    except Exception as e:
                        return GenieResult(
                            space_id=space_id,
                            conversation_id=conversation_id,
                            question=message,
                            content=resp["attachments"][0].get("text", {}).get("content", None)
                        )
                    break
    
                elif status == "EXECUTING_QUERY":
                    self._workspace_client.get(
                        self._make_url(
                            f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}/query-result"),
                        headers={"Content-Type": "application/json"},
                    )
                elif status in ["FAILED", "CANCELED"]:
                    return GenieResult(
                        space_id=space_id,
                        conversation_id=conversation_id,
                        question=message,
                        content=None,
                        error=f"Consulta falhou com status {status}"
                    )
                elif status != "COMPLETED" and attempt < self.max_retries - 1:
                    time.sleep(self.retry_delay)
                else:
                    return GenieResult(
                        space_id=space_id,
                        conversation_id=conversation_id,
                        question=message,
                        content=None,
                        error=f"Consulta falhou ou ainda em execução após {self.max_retries * self.retry_delay} segundos"
                    )
                attempt += 1
            resp = self._workspace_client.get(
                self._make_url(
                    f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}/query-result"),
                headers={"Content-Type": "application/json"},
            )
            resp = resp.json()
            columns = resp["statement_response"]["manifest"]["schema"]["columns"]
            header = [str(col["name"]) for col in columns]
            rows = []
            output = resp["statement_response"]["result"]
            if not output:
                return GenieResult(
                    space_id=space_id,
                    conversation_id=conversation_id,
                    question=message,
                    content=content,
                    sql_query=query,
                    sql_query_description=query_description,
                    sql_query_result=pd.DataFrame([], columns=header),
                )
            for item in resp["statement_response"]["result"]["data_typed_array"]:
                row = []
                for column, value in zip(columns, item["values"]):
                    type_name = column["type_name"]
                    str_value = value.get("str", None)
                    if str_value is None:
                        row.append(None)
                        continue
                    match type_name:
                        case "INT" | "LONG" | "SHORT" | "BYTE":
                            row.append(int(str_value))
                        case "FLOAT" | "DOUBLE" | "DECIMAL":
                            row.append(float(str_value))
                        case "BOOLEAN":
                            row.append(str_value.lower() == "true")
                        case "DATE":
                            row.append(datetime.strptime(str_value, "%Y-%m-%d").date())
                        case "TIMESTAMP":
                            row.append(datetime.strptime(str_value, "%Y-%m-%d %H:%M:%S"))
                        case "BINARY":
                            row.append(bytes(str_value, "utf-8"))
                        case _:
                            row.append(str_value)
                rows.append(row)
    
            query_result = pd.DataFrame(rows, columns=header)
            return GenieResult(
                space_id=space_id,
                conversation_id=conversation_id,
                question=message,
                content=content,
                sql_query=query,
                sql_query_description=query_description,
                sql_query_result=query_result,
            )
    
    
    assert databricks_host is not None, "host não está definido"
    assert databricks_token is not None, "token não está definido"
    assert space_id is not None, "space_id não está definido"
    assert question is not None, "question não está definida"
    assert contextual_history is not None, "contextual_history não está definido"
    client = GenieClient(host=databricks_host, token=databricks_token)
    conversation_id = client.start(space_id)
    if isinstance(conversation_id, str) is False:
        return conversation_id
    formatted_message = f"""Use o histórico contextual para responder à pergunta. O histórico pode ou não ser útil. Use-o se considerar relevante.
    
    Histórico Contextual: {contextual_history}
    
    Pergunta a responder: {question}
    """
    
    try:
        result = client.ask(space_id, conversation_id, formatted_message)
    except Exception as e:
        return f"Erro: {str(e)}"
    return result.to_string_results()

$$;

### Antes de criar `chat_with_sales`

Substitua na célula SQL abaixo:

1. **`'host'`** — URL base do workspace (ex.: `https://xxx.cloud.databricks.com`), **sem** barra no final.
2. **`'token'`** — PAT com permissão para a API Genie; **não versionize** tokens. Em sala de aula, use o método indicado pelo instrutor (segredo, variável de ambiente, etc.).
3. **`'genie_id'`** — ID do espaço Genie de **vendas** criado no passo anterior.

Após o treinamento, migre credenciais para [Databricks Secrets](https://docs.databricks.com/security/secrets/index.html).


In [0]:
%sql
    
USE CATALOG dbacademy;
USE SCHEMA <seu_database>;
CREATE OR REPLACE FUNCTION chat_with_sales(question STRING COMMENT "a pergunta a ser feita sobre dados de e-commerce",
                  contextual_history STRING COMMENT "forneça histórico relevante para poder responder esta pergunta, assuma que o genie não mantém histórico. Use 'sem histórico relevante' se não houver nada relevante para responder a pergunta.")
RETURNS STRING
LANGUAGE SQL
COMMENT 'Este é um agente com o qual você pode conversar para obter respostas a perguntas sobre dados de e-commerce. Tente fazer perguntas simples e forneça histórico se houve conversas anteriores.' 
RETURN SELECT _genie_query(
  'host',
  'token',
  'genie_id',
  question, -- recuperado da função
  contextual_history -- recuperado da função
);